# MCAT mit bidirektionaler Co-Attention (nur Architektur)

Erweiterung von MCAT (Chen et al., ICCV 2021) um eine zweite Co-Attention-Richtung, in der die **Histologie die Genomik führt**.
Das Notebook enthält nur das Modell. Es hat dieselbe Schnittstelle wie der `MCAT`-Wrapper in `models/MCAT/MCAT.ipynb`
(`forward(h, omics, loss_fn, label) -> (results_dict, None)`) und kann deshalb ohne Änderungen an `train_one_epoch` / `evaluate` verwendet werden.

| Schritt | Richtung 1: Gene → Patches (wie MCAT) | Richtung 2: Histologie → Gene (neu) |
|---|---|---|
| Queries | 6 Signatur-Embeddings (je ein SNN pro Signatur) `[6, 256]` | K' Cluster-Tokens (globale KMeans-Prototypen, Gated Pooling pro Cluster) `[K', 256]` |
| Keys / Values | M Patch-Embeddings `[M, 256]` | G Gen-Tokens (ein Token pro eindeutigem Gen) `[G, 256]` |
| Output | 6 genomik-geführte WSI-Tokens `[6, 256]` | K' histologie-geführte Genomik-Tokens `[K', 256]` |
| Danach | Path-Transformer → Gated-Attention-Pooling `[256]` | Omic-Transformer → Gated-Attention-Pooling `[256]` |

Fusion (concat → MLP) → Klassifikator → 4 Logits (diskrete Hazards, wie MCAT).

**Designentscheidungen**
- **Beide Co-Attentions haben wenige Queries.** Die Transformer laufen nur über 6 bzw. K' Tokens, es gibt also kein O(M²) über alle Patches.
- **Gen-Tokens:** $t_j = x_j \cdot w_j + e_j + \sum_{k \in \text{Kat}(j)} c_k$ (Wert-, Identitäts- und Kategorie-Embedding, Dimension `d_gene`), danach `Linear(d_gene → 256)` und LayerNorm. Gene, die in mehreren Signaturen vorkommen, werden nur **einmal** tokenisiert und bekommen alle ihre Kategorie-Embeddings. Beim Training wird zufällig ein Anteil der Gen-Tokens ausgeblendet (`gene_token_dropout`).
- **Cluster:** Ein **globales** KMeans pro Fold, nur auf den Trainings-Slides (`fit_histology_prototypes`). Die Zentren liegen als Buffer im Modell und werden mit dem `state_dict` gespeichert. Die Zuordnung der Patches zu den Clustern ist fest (Kosinus auf den rohen Features). Gelernt wird nur das Pooling innerhalb der Cluster. Leere Cluster werden weggelassen, deshalb K' ≤ K.
- **Cluster-Tokens** = Gated-Attention-Pooling im Cluster + Cluster-Identitäts-Embedding + Embedding von $\log(n_k / M)$, damit die Clustergröße nicht verloren geht.
- Co-Attention mit `torch.nn.MultiheadAttention` (1 Head, wie MCAT). Die Attention-Gewichte sind Werte **nach der Softmax**.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans

## Bausteine
SNN-Block, Gated Attention und Transformer wie im MCAT-Repo (`models/model_utils.py`, `models/model_coattn.py`), hier ohne Abhängigkeit vom Repo.

In [ ]:
def snn_block(dim_in: int, dim_out: int, dropout: float = 0.25) -> nn.Sequential:
    """Linear + ELU + AlphaDropout (Self-Normalizing Network), wie SNN_Block im MCAT-Repo."""
    return nn.Sequential(nn.Linear(dim_in, dim_out), nn.ELU(), nn.AlphaDropout(p=dropout))


class AttnNetGated(nn.Module):
    """Gated Attention (Ilse et al., 2018) wie Attn_Net_Gated im MCAT-Repo. [N, dim] -> Logits [N, 1]."""
    def __init__(self, dim: int = 256, hidden: int = 256, dropout: float = 0.25):
        super().__init__()
        self.attention_a = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Dropout(dropout))
        self.attention_b = nn.Sequential(nn.Linear(dim, hidden), nn.Sigmoid(), nn.Dropout(dropout))
        self.attention_c = nn.Linear(hidden, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.attention_c(self.attention_a(x) * self.attention_b(x))


class AttnPool(nn.Module):
    """Gated-Attention-Pooling + rho, wie path/omic_attention_head + path/omic_rho in MCAT. [L, dim] -> [dim]."""
    def __init__(self, dim: int = 256, dropout: float = 0.25):
        super().__init__()
        self.attn = AttnNetGated(dim, dim, dropout)
        self.rho = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Dropout(dropout))

    def forward(self, x: torch.Tensor):
        A = torch.softmax(self.attn(x).squeeze(-1), dim=0)   # [L]
        return self.rho(A @ x), A


def make_transformer(dim: int = 256, nhead: int = 8, dim_ff: int = 512, dropout: float = 0.25, num_layers: int = 2) -> nn.TransformerEncoder:
    """2-Layer-Transformer-Encoder wie path/omic_transformer in MCAT (hier batch_first)."""
    layer = nn.TransformerEncoderLayer(d_model=dim, nhead=nhead, dim_feedforward=dim_ff, dropout=dropout,
                                       activation="relu", batch_first=True)
    return nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)

## Gen-Tokenizer
Bekommt die 6 Signatur-Vektoren, die der Dataloader in `MCAT.ipynb` liefert, und baut daraus ein Token pro **eindeutigem** Gen.
`omic_groups` ist die Ausgabe von `get_omic_groups(...)`, also 6 Listen mit Spaltennamen. Über sie wird festgelegt, an welcher Stelle der
aneinandergehängten Signatur-Vektoren jedes Gen steht und zu welchen Kategorien es gehört.

In [ ]:
class GeneTokenizer(nn.Module):
    """t_j = x_j * w_j + e_j + sum_{k in Kat(j)} c_k  ->  Linear(d_gene -> dim) + LayerNorm.  Ausgabe: [G, dim]."""
    def __init__(self, omic_groups: list, dim: int = 256, d_gene: int = 64, token_dropout: float = 0.1):
        super().__init__()
        flat = [name for group in omic_groups for name in group]          # Reihenfolge = torch.cat(omics)
        self.gene_names = sorted(set(flat))                               # eindeutige Gene
        first_pos = {}
        for i, name in enumerate(flat):
            first_pos.setdefault(name, i)
        idx_of = {name: j for j, name in enumerate(self.gene_names)}

        cat_mask = torch.zeros(len(self.gene_names), len(omic_groups))    # multi-hot Kategorie-Zugehoerigkeit
        for k, group in enumerate(omic_groups):
            for name in group:
                cat_mask[idx_of[name], k] = 1.0

        self.register_buffer("gather_idx", torch.tensor([first_pos[n] for n in self.gene_names], dtype=torch.long))
        self.register_buffer("cat_mask", cat_mask)

        G = len(self.gene_names)
        self.value_emb = nn.Parameter(torch.randn(G, d_gene) * 0.02)      # w_j: wie der Expressionswert wirkt
        self.id_emb = nn.Parameter(torch.randn(G, d_gene) * 0.02)         # e_j: welches Gen
        self.cat_emb = nn.Parameter(torch.randn(len(omic_groups), d_gene) * 0.02)  # c_k: welche Signatur
        self.proj = nn.Sequential(nn.Linear(d_gene, dim), nn.LayerNorm(dim))
        self.token_dropout = token_dropout

    @property
    def n_genes(self) -> int:
        return len(self.gene_names)

    def forward(self, omics: list):
        """omics: 6 Tensoren [d_i] -> Tokens [G, dim], key_padding_mask [G] (True = ignorieren) oder None."""
        x = torch.cat(omics, dim=-1)[self.gather_idx]                                         # [G]
        tok = x.unsqueeze(-1) * self.value_emb + self.id_emb + self.cat_mask @ self.cat_emb   # [G, d_gene]
        tok = self.proj(tok)                                                                  # [G, dim]
        pad_mask = None
        if self.training and self.token_dropout > 0:
            pad_mask = torch.rand(tok.size(0), device=tok.device) < self.token_dropout
        return tok, pad_mask

## Histologie-Queries aus globalen Clustern
`fit_histology_prototypes` wird **einmal pro Fold** auf den Trainings-Slides aufgerufen, das Ergebnis geht über `model.set_prototypes(...)` ins Modell.
Pro Slide werden höchstens `max_patches_per_slide` Patches gezogen, damit große Slides das Clustering nicht dominieren.

In [ ]:
@torch.no_grad()
def fit_histology_prototypes(slide_features, n_clusters: int = 8, max_patches_per_slide: int = 512,
                             seed: int = 0, n_init: int = 5) -> torch.Tensor:
    """Globales KMeans (Kosinus, d.h. auf L2-normalisierten Features) ueber die Patches aller Trainings-Slides.
    slide_features: Iterable von Tensoren [M_i, in_dim] (rohe Patch-Features, z.B. CONCH v1.5).
    Rueckgabe: L2-normalisierte Zentren [n_clusters, in_dim] (CPU)."""
    gen = torch.Generator().manual_seed(seed)
    sample = []
    for feats in slide_features:
        feats = feats.detach().reshape(-1, feats.size(-1)).float().cpu()
        if feats.size(0) > max_patches_per_slide:
            feats = feats[torch.randperm(feats.size(0), generator=gen)[:max_patches_per_slide]]
        sample.append(F.normalize(feats, dim=-1))
    X = torch.cat(sample).numpy()
    km = KMeans(n_clusters=n_clusters, n_init=n_init, random_state=seed).fit(X)
    return F.normalize(torch.from_numpy(km.cluster_centers_).float(), dim=-1)


class ClusterQueries(nn.Module):
    """Patches -> naechster Prototyp (fest) -> Gated-Attention-Pooling pro Cluster (gelernt)
    + Cluster-Identitaets-Embedding + Embedding von log(n_k / M). Leere Cluster werden weggelassen. Ausgabe: [K', dim]."""
    def __init__(self, n_clusters: int, in_dim: int, dim: int = 256, dropout: float = 0.25):
        super().__init__()
        self.register_buffer("prototypes", torch.zeros(n_clusters, in_dim))
        self.register_buffer("fitted", torch.tensor(False))
        self.attn = AttnNetGated(dim, dim, dropout)
        self.cluster_emb = nn.Embedding(n_clusters, dim)
        self.size_emb = nn.Linear(1, dim)

    @torch.no_grad()
    def set_prototypes(self, centroids: torch.Tensor):
        self.prototypes.copy_(F.normalize(centroids.float(), dim=-1).to(self.prototypes.device))
        self.fitted.fill_(True)

    @torch.no_grad()
    def assign(self, x_raw: torch.Tensor) -> torch.Tensor:
        return (F.normalize(x_raw.float(), dim=-1) @ self.prototypes.T).argmax(dim=-1)   # [M]

    def forward(self, x_raw: torch.Tensor, h: torch.Tensor):
        """x_raw: [M, in_dim] rohe Features (nur fuer die Zuordnung), h: [M, dim] Patch-Embeddings."""
        if not bool(self.fitted):
            raise RuntimeError("Keine Prototypen gesetzt: zuerst model.set_prototypes(fit_histology_prototypes(...)) aufrufen.")
        ids = self.assign(x_raw)                                                         # [M]
        K = self.prototypes.size(0)
        member = ids.unsqueeze(0) == torch.arange(K, device=ids.device).unsqueeze(1)      # [K, M]
        counts = member.sum(dim=1)
        present = counts > 0
        cid = torch.nonzero(present).squeeze(-1)                                         # [K'] belegte Cluster

        logits = self.attn(h).squeeze(-1).unsqueeze(0).expand(K, -1)                      # [K, M]
        logits = logits.masked_fill(~member, float("-inf"))[present]                     # [K', M]
        A = torch.softmax(logits, dim=-1)                                                # Softmax je Cluster
        pooled = A @ h                                                                   # [K', dim]

        frac = counts[present].float().unsqueeze(-1) / ids.numel()                       # [K', 1]
        q = pooled + self.cluster_emb(cid) + self.size_emb(torch.log(frac))
        info = {"cluster_ids": ids, "clusters_present": cid, "cluster_frac": frac.squeeze(-1), "cluster_pool": A}
        return q, info

## Modell

In [ ]:
class MCATBidirectional(nn.Module):
    """MCAT mit bidirektionaler Co-Attention.
    Richtung 1 (wie MCAT): 6 Signatur-Embeddings fragen alle M Patches  -> 6 WSI-Tokens  -> Path-Zweig.
    Richtung 2 (neu):      K' Cluster-Tokens fragen alle G Gen-Tokens    -> K' Omic-Tokens -> Omic-Zweig.

    Schnittstelle wie der MCAT-Wrapper in MCAT.ipynb (Batchgroesse 1):
        h: [1, M, in_dim], omics: Liste aus 6 Tensoren [1, d_i]
        forward(h, omics, loss_fn=None, label=None, return_attention=False) -> (results_dict, None)
    Vor dem Training pro Fold: model.set_prototypes(fit_histology_prototypes(...))."""

    def __init__(self, omic_groups: list, in_dim: int = 768, num_classes: int = 4, n_clusters: int = 8,
                 dim: int = 256, d_gene: int = 64, gene_token_dropout: float = 0.1, fusion: str = "concat",
                 dropout: float = 0.25, coattn_heads: int = 1):
        super().__init__()
        if fusion != "concat":
            raise NotImplementedError("Bisher nur fusion='concat'.")
        omic_sizes = [len(g) for g in omic_groups]

        # Patch-Embedding (wie wsi_net in MCAT)
        self.wsi_net = nn.Sequential(nn.Linear(in_dim, dim), nn.ReLU(), nn.Dropout(dropout))
        # Ein SNN pro Signatur (wie sig_networks in MCAT, model_size_omic='small') -> Queries fuer Richtung 1
        self.sig_networks = nn.ModuleList(
            [nn.Sequential(snn_block(d, dim, dropout), snn_block(dim, dim, dropout)) for d in omic_sizes])
        # Gen-Tokens (Keys/Values) und Cluster-Queries fuer Richtung 2
        self.gene_tokenizer = GeneTokenizer(omic_groups, dim, d_gene, gene_token_dropout)
        self.cluster_queries = ClusterQueries(n_clusters, in_dim, dim, dropout)

        # Co-Attention in beide Richtungen
        self.coattn_g2p = nn.MultiheadAttention(dim, coattn_heads, batch_first=True)   # Gene -> Patches
        self.coattn_p2g = nn.MultiheadAttention(dim, coattn_heads, batch_first=True)   # Histologie -> Gene

        # Je Zweig Transformer + Gated-Attention-Pooling
        self.path_transformer = make_transformer(dim, dropout=dropout)
        self.omic_transformer = make_transformer(dim, dropout=dropout)
        self.path_pool = AttnPool(dim, dropout)
        self.omic_pool = AttnPool(dim, dropout)

        # Fusion + Survival-Kopf
        self.mm = nn.Sequential(nn.Linear(2 * dim, dim), nn.ReLU(), nn.Linear(dim, dim), nn.ReLU())
        self.classifier = nn.Linear(dim, num_classes)

    def set_prototypes(self, centroids: torch.Tensor):
        self.cluster_queries.set_prototypes(centroids)

    def forward(self, h, omics, loss_fn=None, label=None, return_attention: bool = False):
        assert h.dim() == 2 or h.size(0) == 1, "Batchgroesse 1 erwartet (wie MCAT)."
        x_raw = h.reshape(-1, h.size(-1)).float()                     # [M, in_dim]
        omics = [o.reshape(-1).float() for o in omics]                # 6 x [d_i]
        h_path = self.wsi_net(x_raw)                                  # [M, dim]

        # Richtung 1: Gene -> Patches (MCAT-GCA)
        h_sig = torch.stack([net(o) for net, o in zip(self.sig_networks, omics)])      # [6, dim]
        path_tok, A_g2p = self.coattn_g2p(h_sig[None], h_path[None], h_path[None])     # [1, 6, dim], [1, 6, M]

        # Richtung 2: Histologie -> Gene
        q_hist, cl_info = self.cluster_queries(x_raw, h_path)                          # [K', dim]
        gene_tok, gene_mask = self.gene_tokenizer(omics)                               # [G, dim]
        omic_tok, A_p2g = self.coattn_p2g(
            q_hist[None], gene_tok[None], gene_tok[None],
            key_padding_mask=None if gene_mask is None else gene_mask[None])           # [1, K', dim], [1, K', G]

        # Zweige: Transformer + Pooling
        h_p, A_path = self.path_pool(self.path_transformer(path_tok).squeeze(0))       # [dim], [6]
        h_o, A_omic = self.omic_pool(self.omic_transformer(omic_tok).squeeze(0))       # [dim], [K']

        # Fusion + Logits (diskrete Hazards: sigmoid(logits), wie MCAT)
        logits = self.classifier(self.mm(torch.cat([h_p, h_o]))).unsqueeze(0)          # [1, num_classes]

        results_dict = {"logits": logits}
        if loss_fn is not None and label is not None:
            results_dict["loss"] = loss_fn(logits, label)
        if return_attention:
            results_dict["attention"] = {
                "coattn_g2p": A_g2p[0].detach(),     # [6, M]   Signatur -> Patch
                "coattn_p2g": A_p2g[0].detach(),     # [K', G]  Cluster -> Gen (Gen-Namen: model.gene_tokenizer.gene_names)
                "path_pool": A_path.detach(),        # [6]
                "omic_pool": A_omic.detach(),        # [K']
                **{k: v.detach() for k, v in cl_info.items()},
            }
        return results_dict, None

## Smoke-Test mit Zufallsdaten
Prüft Shapes, Backward und Attention-Normierung. Wird übersprungen, wenn vor dem Einbinden `RUN_SMOKE_TEST = False` gesetzt ist (siehe unten).

In [ ]:
def _smoke_test(seed: int = 0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    in_dim, sizes = 768, [12, 30, 40, 35, 120, 38]
    pool = [f"G{i}_rnaseq" for i in range(240)]
    groups = [sorted(rng.choice(pool, s, replace=False).tolist()) for s in sizes]   # mit Ueberschneidungen

    model = MCATBidirectional(groups, in_dim=in_dim, n_clusters=8)
    slides = [torch.randn(int(rng.integers(300, 2000)), in_dim) for _ in range(6)]
    model.set_prototypes(fit_histology_prototypes(slides, n_clusters=8, max_patches_per_slide=256, seed=seed))

    h = slides[0][None]                                        # [1, M, in_dim]
    omics = [torch.randn(1, s) for s in sizes]                 # 6 x [1, d_i]
    loss_fn = lambda logits, label: F.cross_entropy(logits, label[:, 0])   # Platzhalter fuer NLLSurvLoss
    label = torch.tensor([[2, 0]])                             # [time_bin, censorship]

    model.train()
    out, _ = model(h=h, omics=omics, loss_fn=loss_fn, label=label)
    out["loss"].backward()
    no_grad = [n for n, p in model.named_parameters() if p.grad is None]
    assert not no_grad, f"Parameter ohne Gradient: {no_grad}"

    model.eval()
    with torch.no_grad():
        out, _ = model(h=h, omics=omics, return_attention=True)
    att = out["attention"]
    for k in ("coattn_g2p", "coattn_p2g", "cluster_pool"):
        assert torch.allclose(att[k].sum(-1), torch.ones(att[k].size(0))), k

    n_params = sum(p.numel() for p in model.parameters())
    print(f"M = {h.size(1)} Patches | G = {model.gene_tokenizer.n_genes} eindeutige Gene (Summe Signaturen: {sum(sizes)})")
    print(f"logits {tuple(out['logits'].shape)} | g2p {tuple(att['coattn_g2p'].shape)} | p2g {tuple(att['coattn_p2g'].shape)}"
          f" | belegte Cluster {att['clusters_present'].tolist()}")
    print(f"Parameter: {n_params:,} (davon Gen-Tokenizer: {sum(p.numel() for p in model.gene_tokenizer.parameters()):,})")
    print("Smoke-Test ok.")


if globals().get("RUN_SMOKE_TEST", True):
    _smoke_test()

## Einbau in ein Trainingsnotebook (Vorlage: `models/MCAT/MCAT.ipynb`)

**1. Modell laden.** Die Zelle mit dem Import von `MCAT_Surv` und dem `MCAT`-Wrapper ersetzen durch (Pfad relativ zum Trainingsnotebook anpassen):
```python
RUN_SMOKE_TEST = False
%run -i ../../own_models/mcat_bidirectional_coattention.ipynb
```
Dieses Notebook definiert keine globalen Namen wie `config`, `device` oder `dataset`, mit `-i` wird also nichts überschrieben.

**2. Config ergänzen:**
```python
config.update({
    "model_type": "mcat_bidirectional",
    "n_clusters": 8,                  # globale KMeans-Cluster (Histologie-Queries)
    "proto_patches_per_slide": 512,   # Patches pro Slide fuer das KMeans
    "d_gene": 64,                     # Dimension der Gen-Embeddings vor der Projektion auf 256
    "gene_token_dropout": 0.1,        # Anteil ausgeblendeter Gen-Tokens im Training
})
```

**3. In `run_fold` die Modell-Erzeugung ersetzen** (nach `train_ds = ...`):
```python
model = MCATBidirectional(
    omic_groups=omic_groups,
    in_dim=config["in_dim"],
    num_classes=config["num_classes"],
    n_clusters=config["n_clusters"],
    d_gene=config["d_gene"],
    gene_token_dropout=config["gene_token_dropout"],
    fusion=config["fusion"],
).to(dev)

# Prototypen NUR aus den Trainings-Slides dieses Folds (kein Leakage)
centroids = fit_histology_prototypes(
    (train_ds[i][0] for i in range(len(train_ds))),
    n_clusters=config["n_clusters"],
    max_patches_per_slide=config["proto_patches_per_slide"],
    seed=config["seed"] + fold,
)
model.set_prototypes(centroids)
```
`train_one_epoch`, `evaluate`, `NLLSurvLoss`, Dataset und Logger bleiben unverändert.

**Attention auslesen** (z.B. nach dem Training für Heatmaps):
```python
out, _ = model(h=features, omics=omics, return_attention=True)
out["attention"]["coattn_p2g"]          # [K', G]: welches Cluster schaut auf welches Gen
model.gene_tokenizer.gene_names         # Spaltennamen zu den G Gen-Tokens
```